# 02 — Federated Partitioning & Preprocessing Validation

**Stage 2: Feature Verification, Non-IID Dirichlet Partitioning, and Leak-Free Preprocessing**

This notebook reviews and validates the processed federated partitions and verification audits produced in Stage 2.

## 0. Environment Setup

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Project root: {root}")

## 1. Load Pre-Partitioning Verification Reports

In [ ]:
metrics_dir = root / "outputs" / "metrics"
with open(metrics_dir / "verification_report.json") as f:
    v_report = json.load(f)

print("=== A. REDUNDANT PAIRS CHECK ===")
for p in v_report["redundant_pair_check"]["pair_results"]:
    print(f"  {p['feature_a']} vs {p['feature_b']}:")
    print(f"    Mismatch rows: {p['mismatch_rows']:,} / {p['total_rows_checked']:,}")
    print(f"    Decision: {p['decision']}")

print(f"
Dropped columns: {v_report['dropped_columns']}")
print(f"Final feature dimensionality: {v_report['final_feature_count']}")

print("
=== B. DUPLICATE FEATURE-VECTORS & LABEL-CONFLICT AUDIT ===")
dup = v_report["duplicate_vector_check"]
print(f"  Total rows scanned: {dup['total_rows']:,}")
print(f"  Duplicate feature vectors: {dup['duplicate_feature_vector_rows']:,}")
print(f"  Unique feature vectors: {dup['unique_feature_vectors']:,}")
print(f"  Label conflict vectors: {dup['label_conflict_vectors']}")
print(f"  Label conflict rows: {dup['label_conflict_rows']}")
print(f"  Recommendation: {dup['recommendation']}")

## 2. Load Federated Partition Summary (alpha=0.5, seed=42)

In [ ]:
with open(metrics_dir / "partition_summary_alpha0.5_seed42.json") as f:
    p_summary = json.load(f)

print(f"Alpha: {p_summary['alpha']} | Seed: {p_summary['seed']} | Num Clients: {p_summary['num_clients']}")
print(f"Total rows partitioned: {p_summary['total_rows']:,}")
print(f"Validation status: {p_summary['validation']['status']}")
for chk in p_summary['validation']['checks_performed']:
    print(f"  [PASS] {chk}")

## 3. Client Distribution Breakdown

In [ ]:
df_clients = pd.DataFrame([
    {
        "Client": f"Client {c['client_id']:02d}",
        "Total": c["total_samples"],
        "Benign": c["n_benign"],
        "Malicious": c["n_malicious"],
        "Benign_Pct": c["benign_pct"],
        "Train": c["split_sizes"]["train"],
        "Val": c["split_sizes"]["val"],
        "Test": c["split_sizes"]["test"],
        "Benign_Train": c["benign_train_samples"],
    }
    for c in p_summary["client_stats"]
])
df_clients

## 4. Visualizing Non-IID Partition Skew

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df_clients))
width = 0.35

ax.bar(x - width/2, df_clients["Benign"], width, label="Benign (Label=0)", color="#2ca02c")
ax.bar(x + width/2, df_clients["Malicious"], width, label="Malicious (Label=1)", color="#d62728")

ax.set_xlabel("Client ID")
ax.set_ylabel("Number of Flows (log scale)")
ax.set_yscale("log")
ax.set_title("Non-IID Client Data Partitioning (Dirichlet alpha=0.5, K=10)")
ax.set_xticks(x)
ax.set_xticklabels(df_clients["Client"], rotation=45)
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(root / "outputs" / "figures" / "partition_distribution_alpha0.5.png", dpi=150)
plt.show()

## 5. Verify Client-Level Scaler Parameters (Leak-Free MinMax)

In [ ]:
run_dir = root / "data" / "processed" / "alpha_0.5_seed_42"
with open(run_dir / "client_00" / "scaler_params.json") as f:
    sc_params = json.load(f)

df_scaler = pd.DataFrame({
    "Feature": sc_params["feature_names"],
    "P01": sc_params["p01"],
    "P99": sc_params["p99"],
    "Clip_Min": sc_params["clip_min"],
    "Clip_Max": sc_params["clip_max"]
})
df_scaler.head(10)

## 6. Inspect Saved Processed Arrays (.npz)

In [ ]:
c0_data = np.load(run_dir / "client_00" / "data.npz", allow_pickle=True)
print("Client 00 array shapes:")
for k in c0_data.files:
    arr = c0_data[k]
    print(f"  {k:15s}: shape {str(arr.shape):16s} dtype {arr.dtype}")

print("
X_train value range:")
print(f"  Min: {c0_data['X_train'].min():.4f}, Max: {c0_data['X_train'].max():.4f}")
print(f"  Finite check: {np.all(np.isfinite(c0_data['X_train']))}")